In [ ]:
# Problema: Integrar datos operacionales en SQLite para responder una pregunta analítica reproducible.

"""Crea una fuente relacional independiente y su resultado analítico."""
import csv
import sqlite3
from pathlib import Path

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
SOURCE, DATABASE, OUTPUT = ROOT / "data/sales.csv", ROOT / "data/sales.db", ROOT / "submission/customer_sales.csv"


def build_submission():
    DATABASE.unlink(missing_ok=True)
    with sqlite3.connect(DATABASE) as db, SOURCE.open(newline="", encoding="utf-8") as file:
        db.execute("PRAGMA foreign_keys = ON")
        db.executescript("""CREATE TABLE customers (customer_id TEXT PRIMARY KEY, customer_name TEXT, customer_city TEXT);
CREATE TABLE orders (order_id TEXT PRIMARY KEY, order_date TEXT, customer_id TEXT REFERENCES customers(customer_id));
CREATE TABLE products (product_id TEXT PRIMARY KEY, product_name TEXT, category TEXT, unit_price REAL);
CREATE TABLE order_items (order_id TEXT REFERENCES orders(order_id), product_id TEXT REFERENCES products(product_id), quantity INTEGER, PRIMARY KEY(order_id, product_id));""")
        rows = list(csv.DictReader(file))
        db.executemany("INSERT INTO customers VALUES (?, ?, ?)", {(r["customer_id"], r["customer_name"], r["customer_city"]) for r in rows})
        db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", {(r["product_id"], r["product_name"], r["category"], float(r["unit_price"])) for r in rows})
        db.executemany("INSERT INTO orders VALUES (?, ?, ?)", {(r["order_id"], r["order_date"], r["customer_id"]) for r in rows})
        db.executemany("INSERT INTO order_items VALUES (?, ?, ?)", [(r["order_id"], r["product_id"], int(r["quantity"])) for r in rows])
        result = db.execute("""WITH customer_lines AS (
SELECT c.customer_id, c.customer_name, c.customer_city, o.order_id, i.quantity, i.quantity * p.unit_price AS line_total
FROM customers c JOIN orders o USING(customer_id) JOIN order_items i USING(order_id) JOIN products p USING(product_id))
SELECT customer_id, customer_name, customer_city, COUNT(DISTINCT order_id) AS number_of_orders, SUM(quantity) AS total_units, ROUND(SUM(line_total), 2) AS total_sales
FROM customer_lines GROUP BY customer_id, customer_name, customer_city ORDER BY customer_id""").fetchall()
    with OUTPUT.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file); writer.writerow(["customer_id", "customer_name", "customer_city", "number_of_orders", "total_units", "total_sales"]); writer.writerows(result)


if __name__ == "__main__":
    build_submission()